In [1]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries imported successfully.")

# Loading dataset
df = pd.read_csv(r"C:\Users\user\Desktop\DATA WEB\Medical_Device_Failure_dataset.csv")
print(f"Dataset loaded. Shape: {df.shape}")

Libraries imported successfully.
Dataset loaded. Shape: (4149, 13)


In [2]:
# Data Cleaning
df['Purchase_Date'] = pd.to_datetime(df['Purchase_Date'], format='%d/%m/%Y', errors='coerce')
current_year = datetime.now().year
df['Age_Years'] = current_year - df['Purchase_Date'].dt.year

df['Maintenance_Cost'] = df['Maintenance_Cost'].abs()
df['Downtime'] = df['Downtime'].fillna(df['Downtime'].median())
df['Maintenance_Cost'] = df['Maintenance_Cost'].fillna(df['Maintenance_Cost'].median())

# Derived features
df['Cost_per_Downtime'] = df['Maintenance_Cost'] / (df['Downtime'] + 1e-5)
df['Failure_Rate'] = df['Failure_Event_Count'] / (df['Age_Years'] + 1)
df['High_Maintenance_Flag'] = (df['Maintenance_Frequency'] > df['Maintenance_Frequency'].quantile(0.75)).astype(int)

print("Structured feature engineering completed.")

# NLP Feature Extraction
def extract_failure_features(text):
    if not isinstance(text, str) or pd.isna(text):
        return {}
    text_lower = text.lower()
    features = {
        'overheating': len(re.findall(r'overheat|overheating', text_lower)),
        'voltage_spike': len(re.findall(r'voltage.?spike', text_lower)),
        'battery_wear': len(re.findall(r'battery.?wear', text_lower)),
        'software_crash': len(re.findall(r'software.?crash|system.?crash', text_lower)),
        'sensor_misalignment': len(re.findall(r'sensor.?misalign', text_lower)),
        'data_lag': len(re.findall(r'data.?lag|data.?loss', text_lower)),
        'circuit_failure': len(re.findall(r'circuit.?failure', text_lower)),
        'abnormal_sound': len(re.findall(r'abnormal.?sound', text_lower)),
        'critical': len(re.findall(r'critical|emergency|urgent', text_lower)),
    }
    features['report_length'] = len(text_lower.split())
    return features

nlp_features = df['Maintenance_Report'].apply(extract_failure_features)
nlp_df = pd.json_normalize(nlp_features).add_prefix('nlp_')

df = pd.concat([df.reset_index(drop=True), nlp_df.reset_index(drop=True)], axis=1)
print(f"Final dataframe shape: {df.shape}")

# Target: Binary failure risk (high if Failure_Event_Count >= 3)
df['high_risk'] = (df['Failure_Event_Count'] >= 3).astype(int)

# Feature columns
feature_cols = ['Age_Years', 'Maintenance_Cost', 'Downtime', 'Maintenance_Frequency', 
                'Failure_Rate', 'High_Maintenance_Flag'] + [col for col in df.columns if col.startswith('nlp_')]

X = df[feature_cols].fillna(0)
y = df['high_risk']
device_types = df['Device_Type']

print("Features and target prepared.")

Structured feature engineering completed.
Final dataframe shape: (4149, 27)
Features and target prepared.


In [3]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split (stratified by device type for realistic transfer learning)
X_train, X_test, y_train, y_test, dev_train, dev_test = train_test_split(
    X_scaled, y, device_types, test_size=0.25, random_state=42, stratify=y)

# Convert to tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

# Simple Neural Network with transfer-friendly architecture
class FailurePredictor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 1)
        self.dropout = nn.Dropout(0.3)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.relu(self.fc3(x))
        x = self.sigmoid(self.fc4(x))
        return x

input_dim = X_train.shape[1]
model = FailurePredictor(input_dim)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 80
batch_size = 64
train_dataset = torch.utils.data.TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

print("Starting training...")
for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/len(train_loader):.4f}")

# Evaluation
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_t)
    test_preds = (test_outputs.squeeze() > 0.5).numpy().astype(int)
    test_probs = test_outputs.squeeze().numpy()

print("\n=== Transfer Learning Model Performance ===")
print(classification_report(y_test, test_preds, digits=4))
print(f"ROC-AUC Score: {roc_auc_score(y_test, test_probs):.4f}")

# Cross-Device Performance Analysis
results = []
for dev_type in dev_test.unique():
    mask = dev_test == dev_type
    if mask.sum() > 10:  # only meaningful groups
        auc = roc_auc_score(y_test[mask], test_probs[mask]) if len(np.unique(y_test[mask])) > 1 else 0.5
        results.append({'Device_Type': dev_type, 'Samples': mask.sum(), 'ROC_AUC': auc})

cross_device_df = pd.DataFrame(results)
print("\nCross-Device Performance:")
print(cross_device_df.sort_values('ROC_AUC', ascending=False).round(4))

# Save model and results
torch.save(model.state_dict(), 'cross_device_failure_model.pth')
print("\nModel saved as 'cross_device_failure_model.pth'")

Starting training...
Epoch 20/80, Loss: 0.0163
Epoch 40/80, Loss: 0.0048
Epoch 60/80, Loss: 0.0044
Epoch 80/80, Loss: 0.0004

=== Transfer Learning Model Performance ===
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000       722
           1     1.0000    1.0000    1.0000       316

    accuracy                         1.0000      1038
   macro avg     1.0000    1.0000    1.0000      1038
weighted avg     1.0000    1.0000    1.0000      1038

ROC-AUC Score: 1.0000

Cross-Device Performance:
     Device_Type  Samples  ROC_AUC
0  Defibrillator      186      1.0
1  Infusion Pump      184      1.0
2  X-Ray Machine      170      1.0
3    MRI Scanner      159      1.0
4     CT Scanner      164      1.0
5    PET Scanner      175      1.0

Model saved as 'cross_device_failure_model.pth'


In [5]:
# Few-Shot Learning - Fine-tuning on Rare / New Device Types

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np

print("Starting Few-Shot Learning Analysis...")

# Load the pre-trained model from the cell above
model = FailurePredictor(input_dim=X_train.shape[1])
model.load_state_dict(torch.load('cross_device_failure_model.pth'))
model.eval()

# Identify device types with limited samples for few-shot scenario
device_counts = df['Device_Type'].value_counts()
rare_devices = device_counts[device_counts.between(30, 200)].index.tolist()

print(f"Identified {len(rare_devices)} device types suitable for few-shot evaluation.")

few_shot_results = []

for device in rare_devices:
    device_mask = (df['Device_Type'] == device)
    X_dev = X[device_mask]
    y_dev = y[device_mask]
    
    if len(X_dev) < 30 or len(np.unique(y_dev)) < 2:
        continue
    
    # Split into support set (few examples for fine-tuning) and query set (for evaluation)
    X_support, X_query, y_support, y_query = train_test_split(
        X_dev, y_dev, test_size=0.65, random_state=42, stratify=y_dev)
    
    # Scale using the global scaler
    X_support_scaled = scaler.transform(X_support)
    X_query_scaled = scaler.transform(X_query)
    
    # Convert to PyTorch tensors
    X_support_t = torch.tensor(X_support_scaled, dtype=torch.float32)
    y_support_t = torch.tensor(y_support.values, dtype=torch.float32).unsqueeze(1)
    X_query_t = torch.tensor(X_query_scaled, dtype=torch.float32)
    y_query_t = torch.tensor(y_query.values, dtype=torch.float32).unsqueeze(1)
    
    # Create fresh copy of pre-trained model for fine-tuning
    few_shot_model = FailurePredictor(input_dim=X_train.shape[1])
    few_shot_model.load_state_dict(torch.load('cross_device_failure_model.pth'))
    
    # Freeze early layers - fine-tune only later layers (transfer learning)
    for param in list(few_shot_model.fc1.parameters()) + list(few_shot_model.fc2.parameters()):
        param.requires_grad = False
    
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, few_shot_model.parameters()), lr=0.002)
    criterion = nn.BCELoss()
    
    # Few-shot fine-tuning (limited epochs)
    few_shot_model.train()
    for epoch in range(12):
        optimizer.zero_grad()
        outputs = few_shot_model(X_support_t)
        loss = criterion(outputs, y_support_t)
        loss.backward()
        optimizer.step()
    
    # Evaluation on query set
    few_shot_model.eval()
    with torch.no_grad():
        query_outputs = few_shot_model(X_query_t)
        query_preds = (query_outputs.squeeze() > 0.5).numpy().astype(int)
        query_probs = query_outputs.squeeze().numpy()
    
    auc_fewshot = roc_auc_score(y_query, query_probs) if len(np.unique(y_query)) > 1 else 0.5
    
    few_shot_results.append({
        'Device_Type': device,
        'Samples_Total': len(X_dev),
        'Support_Size': len(X_support),
        'ROC_AUC_FewShot': auc_fewshot
    })

# Create DataFrame
few_shot_df = pd.DataFrame(few_shot_results)

print("\n Few-Shot Learning Results")
if not few_shot_df.empty:
    print(few_shot_df.sort_values('ROC_AUC_FewShot', ascending=False).round(4))
else:
    print("No suitable devices found for few-shot evaluation.")

# Summary
print("\nFew-Shot Learning Summary:")
print(f"Number of devices evaluated: {len(few_shot_df)}")
print(f"Average ROC-AUC (Few-Shot): {few_shot_df['ROC_AUC_FewShot'].mean():.4f}" if not few_shot_df.empty else "N/A")

# Save results
few_shot_df.to_csv('few_shot_transfer_learning_results.csv', index=False)
print("\nFew-shot results saved to 'few_shot_transfer_learning_results.csv'")

Starting Few-Shot Learning Analysis...
Identified 0 device types suitable for few-shot evaluation.

 Few-Shot Learning Results
No suitable devices found for few-shot evaluation.

Few-Shot Learning Summary:
Number of devices evaluated: 0
N/A

Few-shot results saved to 'few_shot_transfer_learning_results.csv'


In [6]:
# Summary and Visualization

import matplotlib.pyplot as plt
import seaborn as sns

print("Generating Final Transfer Learning Summary...")

if 'few_shot_df' in locals() and not few_shot_df.empty:
    plt.figure(figsize=(14, 8))
    few_shot_sorted = few_shot_df.sort_values('ROC_AUC_FewShot', ascending=False).head(12)
    sns.barplot(data=few_shot_sorted, x='ROC_AUC_FewShot', y='Device_Type', palette='viridis')
    plt.title('Few-Shot Learning Performance by Device Type')
    plt.xlabel('ROC-AUC Score')
    plt.ylabel('Device Type')
    plt.xlim(0.5, 1.0)
    plt.grid(axis='x', alpha=0.3)
    plt.show()

    print("\nTop 8 Devices by Few-Shot Performance:")
    print(few_shot_df.sort_values('ROC_AUC_FewShot', ascending=False).head(8)[['Device_Type', 'Samples_Total', 'Support_Size', 'ROC_AUC_FewShot']].round(4))
else:
    print("No few-shot results available.")

print("\n Completed!")

Generating Final Transfer Learning Summary...
No few-shot results available.

 Completed!
